# 6 · Stats  `[EVAL]`

The **full statistical tables** behind the figures in `1`–`3` — kept here so the analysis notebooks stay figure-led. Everything is full-conversation eval, paired by the 96 shared personas; thin arms (<3 scored iters) are dropped to avoid NaN rows. Exports → `results/<VIEW>/tables/6_stats/`. For a reader who wants exact numbers.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, training, pref, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="6_stats",                # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × metric): the FINAL and the BEST iteration vs base in one table (column `target`) — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm), bootstrap 95% CI, trajectory ρ / slope.

In [ ]:
MR  = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"),  S.SCORES)
MRall = pd.concat([MR.assign(target="final"), MRb.assign(target="best")], ignore_index=True)
MRall = MRall[["arm", "rubric", "target", "base", "target_iter"] +
              [c for c in MRall.columns if c not in ("arm", "rubric", "target", "base", "target_iter")]]
display(MRall)
eda_analysis.save_table(MRall, "main_results", caption="FINAL and BEST iteration vs base per arm x metric (column `target`), persona-paired (N=96): dz, Wilcoxon p (Holm within arm x target), bootstrap CI, trajectory rho/slope. Thin arms dropped.")

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm × rubric.

In [3]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
eda_analysis.save_table(FR.round(4), "friedman_omnibus", caption="Friedman repeated-measures omnibus across iterations per arm x rubric (Kendall's W). Thin arms dropped.")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,metric,chi2,p,kendall_w,k_iters,n_personas
0,GRPO_LA0,Q1Q2,312.7165,0.0000,0.3257,11,96
1,GRPO_LA0,WAI-SR,166.3047,0.0000,0.1732,11,96
2,GRPO_LA0,CSQ-8,111.1489,0.0000,0.1158,11,96
3,GRPO_LA0,MI-SAT,129.4716,0.0000,0.1349,11,96
4,GRPO_LA0,MITI,303.5148,0.0000,0.3162,11,96
5,GRPO_LA0,PCT,58.1104,0.0000,0.0605,11,96
6,GRPO_LA0,MICI,335.2027,0.0000,0.3492,11,96
7,GRPO_LA0,Q1,265.6537,0.0000,0.2767,11,96
8,GRPO_LA0,Q2,329.8852,0.0000,0.3436,11,96
9,PTO_LA0,Q1Q2,432.4498,0.0000,0.4505,11,96


'c:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables\\stats'

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN base).

In [ ]:
frames = []
for arm in sorted(S.SCORES.arm.unique()):
    if arm in stats.thin_arms(S.SCORES): continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if not PV.empty: frames.append(PV)
if frames:
    VB = pd.concat(frames, ignore_index=True)[["arm", "iteration", "n", "mean_delta", "dz", "p", "p_holm", "ci_low", "ci_high"]].round(4)
    display(VB)
    eda_analysis.save_table(VB, "vs_base_paired", caption="Each arm x iteration vs its OWN base on Q1+Q2 — one merged table (column `arm`); persona-paired Wilcoxon, dz, Holm p (within arm), bootstrap CI.")
else:
    print("no non-thin arms scored.")

## 4 · Cross-arm contrasts — PTO vs GRPO (matched K) & K0 vs K5  `[EVAL]`
**Purpose.** The method and look-ahead contrasts as paired tables at matched iterations (iteration-0 base rows dropped). Positive `mean_delta` = first term higher.

In [ ]:
frames = []
for K in sorted(S.SCORES.K.unique()):
    CMP = stats.paired_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    if not CMP.empty: frames.append(CMP[CMP.iteration > 0])
if frames:
    MP = pd.concat(frames, ignore_index=True)
    view = MP[["K", "iteration", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO - GRPO at matched K + iterations (+ => PTO higher) ==="); display(view)
    eda_analysis.save_table(view, "method_paired_by_K", caption="PTO - GRPO at matched K and matched iterations — one merged table (column `K`); persona-paired Wilcoxon + dz + Holm. + => PTO higher.")
else:
    print("no common PTO/GRPO iterations at any K.")
frames = []
for method in ["PTO", "GRPO"]:
    CMP = stats.paired_k_comparison(S.SCORES, method)
    if not CMP.empty: frames.append(CMP[CMP.iteration > 0])
if frames:
    KP = pd.concat(frames, ignore_index=True)
    view = KP[["method", "iteration", "metric", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== K0 - K5 within each method (+ => K0 higher) ==="); display(view)
    eda_analysis.save_table(view, "k_paired_by_method", caption="K0 - K5 within each method at matched iterations — one merged table (column `method`); persona-paired Wilcoxon + dz + Holm. + => K0 higher.")
else:
    print("K0-vs-K5 not comparable yet for either method.")

## 5 · Climb rate, rankings, factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm (climb rate vs endpoint); the cross-model leaderboard; and the rubric PCA (PC1 share → are the 6 rubrics ~one factor?).

In [ ]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, m)
      for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS]), S.SCORES)
SLv = SL[["arm", "metric", "spearman_rho", "ols_slope", "peak_iter", "final_iter"]].round(4)
display(SLv)
eda_analysis.save_table(SLv, "slope_by_arm", caption="Per-iteration OLS slope + Spearman rho per arm x metric (climb rate; peak vs final iteration flags a regression). Thin arms dropped.")
PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in sorted(S.SCORES.arm.unique()) if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
eda_analysis.save_table(PCA, "rubric_pca_pc1", caption="Variance explained by PC1 of the rubric scores per arm (dominant PC1 => rubrics ~ one latent factor).")

## 6 · GRPO iter-9 anomaly check  `[EVAL]`
**Purpose.** The all-metric trajectory grid shows GRPO_LA0 dipping at **iter 9 across almost every metric simultaneously**, then partially recovering at 10 (while Q1+Q2 declines at both 9 and 10). Persona-paired deltas it8→9, it9→10, it8→10 test whether iter 9 is a one-iteration dip (eval-noise / transient policy state) vs the start of the regression. **Read:** significant negative it9−it8 followed by positive it10−it9 = a transient dip; monotonic negative it8→10 on Q1+Q2 = the real reward-hack regression.

In [ ]:
ARM = "GRPO_LA0"
a = S.SCORES[S.SCORES.arm == ARM]
need = {8, 9, 10}
if not a.empty and need <= set(a.iteration.unique()):
    model_at = lambda it: a[a.iteration == it].model.iloc[0]
    mets = [m for m in ["Q1Q2", "MITI", "WAI-SR"] if m in S.METRICS]
    CHK = pd.concat([stats.compare_two_models(a, model_at(hi), model_at(lo), mets).assign(contrast=f"it{hi}-it{lo}")
                     for lo, hi in [(8, 9), (9, 10), (8, 10)]], ignore_index=True)
    view = CHK[["contrast", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
    display(view)
    eda_analysis.save_table(view, "grpo_iter9_check",
        caption="GRPO_LA0 iter-9 anomaly: persona-paired deltas it8->9, it9->10, it8->10 on Q1+Q2/MITI/WAI-SR. A significant negative it9-it8 followed by a positive it10-it9 = a one-iteration dip (eval-noise or transient policy state), distinct from the monotonic Q1+Q2 reward-hack regression.")
else:
    print(f"{ARM} iters 8-10 not all scored in this view — anomaly check skipped.")

## 7 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (6_Stats runs last, so this call captures every family).

In [ ]:
print("index ->", eda_analysis.build_index())